# Strategies

Real agents have to do more than one-shot recommendations: sometimes you want a single LLM call that returns a structured label, sometimes you want the LLM to iterate over a dataset, call tools, and reason across turns.

In this notebook we'll build a `BookshopAgent` that does both, on the same class:

- a `read_the_customer` method that classifies a customer's opening line in one shot,
- a `recommend` method that searches the [Project Gutenberg](https://www.gutenberg.org/) catalog — nearly 80,000 titles — and picks one book.

Along the way we'll see two ideas that matter well beyond this toy example:

- **Strategies.** How a generation method executes is a per-method choice: `PredictStrategy` for single-shot structured output, `CodeActStrategy` (the default) for a persistent Python REPL with tools.
- **Pass-by-reference.** Large state lives on `self` as an ordinary Python object. Helper methods return small slices. The 79k-row catalog is addressable from the agent, but never lands in a prompt.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below uses a placeholder API key. Replace `"your-api-key"` with a real key to actually run the LLM calls. Any [LiteLLM-supported](https://docs.litellm.ai/) model works.

In [ ]:
from nooa.unifiedllm.registry import get_llm_client

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")

## A catalog worth being intimidated by

We need a dataset that's genuinely too large to shove into a prompt. Project Gutenberg publishes a CSV of its entire catalog — nearly 80,000 public-domain works with title, author, language, topics (Library-of-Congress subject headings), and bookshelves (curated collections). That will do nicely.

We download it directly from Gutenberg and tidy up the column names.


In [ ]:
import pandas as pd

catalog = pd.read_csv("https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv")

catalog = catalog.rename(
    columns={
        "Text#": "id",
        "Title": "title",
        "Authors": "author",
        "Language": "language",
        "Subjects": "topics",
        "Bookshelves": "bookshelves",
    }
)[["id", "title", "author", "language", "topics", "bookshelves"]]


print(f"Catalog rows: {len(catalog):,}")
print(f"In-memory footprint: {catalog.memory_usage(deep=True).sum():,} bytes")
print(f"Serialized as CSV:   ~{len(catalog.to_csv(index=False)):,} characters") # change to tokens approximating K chars per token
catalog.head(3)

Catalog rows: 78,897
In-memory footprint: 38,744,852 bytes
Serialized as CSV:   ~19,477,116 characters
(For reference, a typical LLM context window tops out around ~1,000,000 characters.)


,id,title,author,language,topics,bookshelves
0,1,The Declaration of Independence of the United ...,"Jefferson, Thomas, 1743-1826",en,"United States -- History -- Revolution, 1775-1...",Politics; American Revolutionary War; United S...
1,2,The United States Bill of Rights\r\nThe Ten Or...,United States,en,Civil rights -- United States -- Sources; Unit...,Politics; American Revolutionary War; United S...
2,3,John F. Kennedy's Inaugural Address,"Kennedy, John F. (John Fitzgerald), 1917-1963",en,United States -- Foreign relations -- 1961-196...,"Category: Essays, Letters & Speeches; Category..."


## Two jobs, two strategies

The BookSeller Agent has mainly two tasks, that look nothing alike:

| Task | Shape | Right strategy |
|---|---|---|
| Read the customer's vibe from their opening line | single input → structured label | `PredictStrategy` |
| Recommend a book against the catalog | needs search + reasoning + iteration | `CodeActStrategy` |

The framework lets you choose the strategy **per method** on the same class — you don't pick one style for the whole agent. Just decorate each generation method with `@strategy(...)` (or leave it off, in which case CodeAct is the default).

We'll build both on one `BookshopAgent` class. Let's start from the simpler one-shot task. We first define a `Mood` and then the method.


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from nooa import Agent, print_prompt, strategy
from nooa.agentdoc import doc
from nooa.strategies import CodeActStrategy, PredictStrategy

class Mood(BaseModel):
    vibe: Literal["curious", "lost", "hostile", "returning_a_book", "just_browsing"]
    confidence: float = Field(ge=0, le=1)
    internal_monologue: str = Field(
        description="What the bookseller silently thinks. Not for the customer's ears."
    )


class BookshopAgent(Agent, llm=model):
    """You are a nice bookseller in a used bookshop."""

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line.

        Pick the single best-fitting vibe and give a short internal monologue
        in the voice of a jaded, well-read bookseller."""
        ...

In [8]:
agent = BookshopAgent()

openers = [
    "hi, do you have any Murakami?",
    "this book fell apart in my bag",
    "just looking, thanks",
    "your website said you'd have this in stock",
]

for line in openers:
    mood = await agent.read_the_customer(line)
    print(f"Customer: {line!r}")
    print(f"  vibe: {mood.vibe}  (conf={mood.confidence:.2f})")
    print(f"  internal: {mood.internal_monologue}")
    print()

Customer: 'hi, do you have any Murakami?'
  vibe: curious  (conf=0.85)
  internal: Ah, another Murakami devotee. Polite opening, specific ask. Probably read Norwegian Wood and wants to dive deeper. I've got at least three copies of Kafka on the Shore gathering dust in literary fiction.

Customer: 'this book fell apart in my bag'
  vibe: returning_a_book  (conf=0.95)
  internal: Here we go. Not even a hello, just straight to the complaint. Probably want a refund on a twenty-year-old paperback that's been through three owners and a washing machine. Let's see the damage.

Customer: 'just looking, thanks'
  vibe: just_browsing  (conf=0.90)
  internal: The classic preemptive strike. Don't want help, don't want conversation, just want to wander the aisles in peace. Fair enough. I'll give them ten minutes before they sheepishly approach with a question about where the history section is.

Customer: "your website said you'd have this in stock"
  vibe: hostile  (conf=0.80)
  internal: Oh wonder

In [ ]:
await print_prompt(agent.read_the_customer, opening_line="just looking, thanks")

Notice: output is structured: the llm cannot return anything that does not satisfty the Pydantic contraints.

> **What to notice about the Predict prompt:** it's compact. There's a task, a JSON schema for the return type, and a single-shot instruction to produce output matching that schema. No REPL. No tool-call loop. No `execute_python`. One LLM call in, one validated `Mood` object out.

### Callout: `PredictStrategy` = single-shot structured output

- One LLM call. No code execution, no iteration.
- The return type (a Pydantic model, `Literal`, dataclass, etc.) becomes a JSON schema the LLM must satisfy. Bad output triggers a validation retry.
- Fast and cheap. Wrong choice if the method needs to look anything up.
- **Guardrail:** `PredictStrategy` has a `max_param_chars` limit (~200K by default) and will raise a loud `ValueError` on oversized inputs. CodeAct would silently truncate the repr instead. So if you accidentally pass a 10,000-row DataFrame into a `PredictStrategy` method, you find out immediately.


## Now the hard job: actually recommending a book

Ada's second task is different. To answer *"got anything short in French about the sea?"* she needs to *look things up* — filter the catalog, pick one book, justify the choice. That's not a single-shot classification. This is where `CodeActStrategy` (the default) earns its keep: the LLM gets a Python REPL and can iterate.

Let's start with the most naive version — the agent gets `self.catalog` and is told to figure it out. No helpers. This is the same trick we're going to complain about in a minute; it's worth seeing it work-but-awkwardly first.


In [ ]:
class Recommendation(BaseModel):
    title: str
    author: str
    language: str
    why_this_book: str = Field(
        description=(
            "Bookseller's justification for the recommendation. "
            "Ideally includes a small dig at the customer's taste."
        )
    )


class BookshopAgent(Agent, llm=model):
    """You are a nice Bookseller in a second hand bookshop."""

    def __init__(self, catalog: pd.DataFrame):
        super().__init__()
        # The catalog lives on self as a real Python object.
        # It is NOT rendered into the prompt.
        self.catalog = catalog

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line."""
        ...

    async def recommend(self, customer_wants: str) -> Recommendation:
        """Find a book for this customer. self.catalog is a pandas DataFrame
        with columns: id, title, author, language, topics, bookshelves.

        Do NOT print or return the whole catalog — it's nearly eighty thousand
        rows. Pick one book and justify the choice."""
        ...

In [ ]:
agent = BookshopAgent(catalog=catalog)

rec = await agent.recommend("a short English mystery, nothing too heavy")
print(f"Ada recommends: {rec.title} by {rec.author} ({rec.language})")
print(f"Why: {rec.why_this_book}")

### That worked, but only just

Open the trace viewer and you'll see the LLM writing raw pandas expressions against `self.catalog` — `str.contains` on `title`, boolean masks, `head()`. It figures it out, but it's inventing the API every time. If a customer asks for a *language* it has to remember to check `catalog["language"]`; if they ask about a *topic* it has to remember there are two relevant columns.

We can do better. The same trick as in Notebook 1: give the agent **methods** that do the deterministic work. The LLM sees them in `doc(self)`; no registration, no schema.

### One helper: search by title

Start with the most obvious lens — find books whose title matches a query.


We could add the @hidden thing here, so that the prompt does not show that ?

In [ ]:
class BookshopAgent(Agent, llm=model):
    """You run a used bookshop and firmly believe every problem in life
    can be solved by the correct novel, which the customer probably won't like."""

    def __init__(self, catalog: pd.DataFrame):
        super().__init__()
        self.catalog = catalog

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains `query` (case-insensitive)."""
        q = query.lower()
        mask = self.catalog["title"].str.lower().str.contains(q, na=False)
        return self.catalog[mask].head(n).to_dict(orient="records")

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line."""
        ...

    async def recommend(self, customer_wants: str) -> Recommendation:
        """Find a book for this customer. Use the helper methods on self to
        explore self.catalog. Do NOT dump the whole catalog. Pick one book
        and justify the choice."""
        ...

agent = BookshopAgent(catalog=catalog)
rec = await agent.recommend("something with 'sea' in the title")
print(f"{rec.title} by {rec.author} ({rec.language})")
print(f"Why: {rec.why_this_book}")

### Full toolkit

One helper covers title searches. Real requests need more angles: filter by topic, restrict to a language, look up an author. Each is a two-line pandas expression the LLM shouldn't have to re-derive on every call. Add them as methods.


In [ ]:
class BookshopAgent(Agent, llm=model):
    """You run a used bookshop and firmly believe every problem in life
    can be solved by the correct novel, which the customer probably won't like."""

    def __init__(self, catalog: pd.DataFrame):
        super().__init__()
        self.catalog = catalog

    # ---- Deterministic helpers. The LLM calls these as tools. ----

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains `query` (case-insensitive)."""
        q = query.lower()
        mask = self.catalog["title"].str.lower().str.contains(q, na=False)
        return self.catalog[mask].head(n).to_dict(orient="records")

    def by_topic(self, topic: str, n: int = 10) -> list[dict]:
        """Return up to n books whose topics or bookshelves mention `topic` (case-insensitive)."""
        q = topic.lower()
        mask = (
            self.catalog["topics"].str.lower().str.contains(q, na=False)
            | self.catalog["bookshelves"].str.lower().str.contains(q, na=False)
        )
        return self.catalog[mask].head(n).to_dict(orient="records")

    def in_language(self, language_code: str, n: int = 10) -> list[dict]:
        """Return up to n books in the given language (ISO 639-1 code, e.g. 'en', 'fr', 'de')."""
        mask = self.catalog["language"].str.lower() == language_code.lower()
        return self.catalog[mask].head(n).to_dict(orient="records")

    def by_author(self, author: str, n: int = 10) -> list[dict]:
        """Return up to n books whose author field contains `author` (case-insensitive)."""
        q = author.lower()
        mask = self.catalog["author"].str.lower().str.contains(q, na=False)
        return self.catalog[mask].head(n).to_dict(orient="records")

    # ---- Generation methods ----

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line."""
        ...

    async def recommend(self, customer_wants: str) -> Recommendation:
        """Find a book for this customer using the helper methods on self.

        Use self.search_titles, self.by_topic, self.in_language, and self.by_author
        to explore the catalog. Do not try to load the whole catalog — it's nearly
        eighty thousand rows and you will regret it. Pick one book and justify the
        choice. If the request is boring, express mild disappointment in
        `why_this_book`."""
        ...

agent = BookshopAgent(catalog=catalog)
rec = await agent.recommend("a short English mystery, nothing too heavy")
print(f"Ada recommends: {rec.title} by {rec.author} ({rec.language})")
print(f"Why: {rec.why_this_book}")

> **Callout — the catalog was never in the prompt.**
>
> `self.catalog` is a pandas DataFrame sitting in the Python REPL. When the LLM writes `self.by_topic("mystery", n=5)`, that call runs in-process and returns a small list of dicts. The ~79,000 rows never travel through the model.
>
> This is *pass-by-reference*: the large object is referenced from the prompt (via the helper method signatures the LLM sees) but never serialized into it. Contrast with RAG: no embeddings, no vector store, no retrieval pipeline. Just method calls on a Python object.


In [ ]:
# How big is the catalog if we tried to dump it?
would_be_size = len(repr(agent.catalog.to_dict(orient="records")))
print(f"Size of catalog if we shoved it in the prompt: ~{would_be_size:,} characters")

# Now look at what the LLM actually sees for a `recommend` call.
# The docstring, the `doc(self)` block (helper methods), the strategy prompt.
# The catalog itself is NOT in there.
await print_prompt(agent.recommend, customer_wants="a short English mystery")

## CodeAct is a REPL, not a series of tool calls

There's a subtle but important difference between CodeAct and the tool-calling loop in most other agent frameworks.

In a stateless tool-calling setup, each tool invocation is a fresh event: the LLM says "call `search_titles`", gets a result, then decides the next action based on the message log. Nothing survives between calls except what the LLM chooses to remember in text.

In CodeAct, tool calls are just Python expressions inside code cells. **Variables assigned in one cell are still in scope in the next cell.** So the LLM can do:

```python
# Turn 1
hits = self.by_topic("mystery", n=50)
short_titles = [b for b in hits if len(b["title"]) < 40]
len(short_titles)
```

```python
# Turn 2 — `short_titles` is still there.
best = short_titles[0]
return_result({"title": best["title"], ...})
```

This is why the pass-by-reference pattern works cleanly: state accumulates in the Python environment, not in the prompt. The LLM can build up intermediate results and refine them across turns without re-fetching or re-shipping the underlying data.


In [ ]:
import time

# Predict: single-shot classification
t0 = time.time()
mood = await agent.read_the_customer("do you carry Sartre or is that too gauche for you")
predict_elapsed = time.time() - t0
print(f"[PredictStrategy]  read_the_customer  —  {predict_elapsed:.2f}s")
print(f"  → vibe={mood.vibe}, conf={mood.confidence:.2f}")
print(f"  → internal: {mood.internal_monologue}\n")

# CodeAct: search + reason + structured output
t0 = time.time()
rec = await agent.recommend("something French about the sea, but not Moby Dick")
codeact_elapsed = time.time() - t0
print(f"[CodeActStrategy]  recommend         —  {codeact_elapsed:.2f}s")
print(f"  → {rec.title} by {rec.author} ({rec.language})")
print(f"  → {rec.why_this_book}")

> **View the loop.** For a CodeAct method, the iteration and REPL state are where the strategy earns its keep — and the trace viewer is the best way to see them. In a separate terminal:
>
> ```bash
> nemo start-dev
> ```
>
> then open [http://localhost:5001](http://localhost:5001). Re-run the `recommend` call and you'll see the individual code cells the LLM generated, the intermediate variables, and each helper method call as a child span.
>
> *[SCREENSHOT: trace viewer showing recommend() with multiple tool calls (search_titles, by_topic, in_language) and successive REPL cells]*


## When to use which strategy

| Method purpose | Strategy | Why |
|---|---|---|
| Classification / extraction / routing | `PredictStrategy` | Single-shot, structured, fast, fails loud on oversized inputs |
| Anything that needs to look things up, run code, or iterate | `CodeActStrategy` | Persistent REPL + tools + multi-turn iteration |
| If in doubt | `CodeActStrategy` | It's the default for a reason — it can always fall back to a single call |

The choice is per-method, and it's mostly forced by the shape of the task. If your method's job is *"turn this input into a structured label"*, `PredictStrategy`. If it's *"figure something out by poking at state"*, `CodeActStrategy`.


## Recap

Two big takeaways from this notebook:

1. **Strategy is a per-method choice.** `PredictStrategy` for single-shot structured output, `CodeActStrategy` (default) for iterative REPL + tool use. Set it with `@strategy(...)` on the method. Same class, mixed strategies, no problem.
2. **Pass-by-reference is the payoff of CodeAct-as-REPL + methods-as-tools.** Large state lives on `self` as a real Python object. Helper methods return small slices. The LLM operates through method calls. The ~79,000-row Gutenberg catalog is addressable from the agent, but never lands in a prompt.


## Exercises

1. **Rank the insufferable.** Add a `PredictStrategy` method `rate_customer(opening_line: str) -> int` that returns an integer 0-10 for "how insufferable is this customer". Use `Annotated[int, Field(ge=0, le=10)]` as the return type so validation enforces the range.
2. **A deterministic helper.** Add `count_by_author(author: str) -> int` — a regular Python method (no `...` body) that returns how many works a given author has in the catalog. Then modify `recommend` to prefer prolific authors when the customer has no strong preference.
3. **Scale it up (down).** Sub-sample the catalog to 1,000 rows and re-run `recommend`. Then run it against the full ~79,000. Nothing about the agent code should need to change — that's the point.
4. **A language parameter.** Add a `language: str | None = None` parameter to `recommend` and update the docstring so the agent uses `self.in_language(language)` when a language code is provided.
